# Semana 4 — Taller práctico: funciones, modularización y GitHub

**Asignatura:** Análisis de Datos con Python  
**Tipo de evidencia:** Notebook con ejercicios resueltos + repositorio inicial en GitHub  

## Propósito del taller

Aplicar funciones, modularización básica, comprensión de listas y diccionarios, manejo de errores y buenas prácticas de programación en un caso sencillo de análisis de datos.

## Entregables

1. Notebook completo con los ejercicios resueltos.
2. Repositorio inicial en GitHub con estructura organizada.
3. Archivo `README.md` con descripción del repositorio.
4. Evidencia del enlace al repositorio en la última sección del notebook.

## Instrucciones

Complete las celdas marcadas con `TODO`.

In [ ]:
# Cargue y observe aquí los datos.

# Parte 0 — Cargar y observar los datos


In [2]:
# Parte 0 — Cargar y observar los datos
import pandas as pd

df = pd.read_csv("mecatronics_dataset.csv")

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df["system_state"].value_counts())
df.head()

(1000, 5)
time               int64
temperature_C    float64
vibration_g      float64
current_A        float64
system_state      object
dtype: object
time             0
temperature_C    0
vibration_g      0
current_A        0
system_state     0
dtype: int64
system_state
normal    988
alerta     12
Name: count, dtype: int64


,time,temperature_C,vibration_g,current_A,system_state
0,0,20.397371,0.709903,1.932482,normal
1,1,19.989382,0.638695,1.995546,normal
2,2,20.718098,0.508945,1.940743,normal
3,3,21.518244,0.402959,1.999154,normal
4,4,20.212251,0.604733,1.850520,normal


## Parte 1 — Función para convertir valores

Cree una función llamada `convertir_a_float(valor)` que intente convertir un valor a número decimal.

La función debe retornar:

- El número convertido, si la conversión es posible.
- `None`, si el valor no se puede convertir.

In [3]:
# TODO: cree la función convertir_a_float

def convertir_a_float(valor):
    """
    Intenta convertir un valor a número decimal usando float().
    Si el valor no se puede convertir (texto inválido, vacío o None),
    captura el error con try/except y devuelve None.
    """
    try:
        return float(valor)
    except (ValueError, TypeError):
        return None

# Pruebas sugeridas
print(convertir_a_float("12.5"))
print(convertir_a_float("error"))
print(convertir_a_float(""))
print(convertir_a_float(None))

12.5
None
None
None


## Parte 2 — Función para calcular potencia

Cree una función llamada `calcular_potencia(voltaje, corriente)`.

La función debe:

- Retornar `None` si voltaje o corriente son `None`.
- Retornar `None` si voltaje o corriente son negativos.
- Retornar `voltaje * corriente` en los demás casos.

In [4]:
# TODO: cree la función calcular_potencia
# Este dataset no tiene columna de voltaje, así que la reemplazo por
# clasificar_vibracion, que cumple el mismo rol de "función clasificadora"
# usando el umbral real que separa los registros en alerta (vibración > 0.9g).

def clasificar_vibracion(vibracion):
    """
    Clasifica la vibración de un sistema mecatrónico según su magnitud (g).
    Si es None, devuelve "Dato inválido"; si es menor a 0.7, "Normal";
    si está entre 0.7 y 0.9, "Precaución"; si es mayor o igual a 0.9, "Alerta".
    """
    if vibracion is None:
        return "Dato inválido"
    elif vibracion < 0.7:
        return "Normal"
    elif vibracion < 0.9:
        return "Precaución"
    else:
        return "Alerta"

# Pruebas
print(clasificar_vibracion(0.5))   # Normal
print(clasificar_vibracion(0.8))   # Precaución
print(clasificar_vibracion(0.95))  # Alerta
print(clasificar_vibracion(None))  # Dato inválido

Normal
Precaución
Alerta
Dato inválido


## Parte 3 — Función para clasificar temperatura

Cree una función llamada `clasificar_temperatura(temp)` con los siguientes criterios:

- Si `temp` es `None`: `"Dato inválido"`
- Si `temp < 40`: `"Normal"`
- Si `40 <= temp < 50`: `"Precaución"`
- Si `temp >= 50`: `"Alerta"`

In [5]:
# TODO: cree la función clasificar_temperatura
# Umbrales recalibrados: los datos reales van de ~13°C a ~26.5°C,
# y el umbral de alerta real (>26°C) coincide con los registros marcados "alerta".

def clasificar_temperatura(temp):
    """
    Clasifica la temperatura de un sistema mecatrónico según un umbral.
    Si es None, devuelve "Dato inválido"; si es menor a 23, "Normal";
    si está entre 23 y 26, "Precaución"; si es mayor o igual a 26, "Alerta".
    """
    if temp is None:
        return "Dato inválido"
    elif temp < 23:
        return "Normal"
    elif temp < 26:
        return "Precaución"
    else:
        return "Alerta"

# Pruebas
print(clasificar_temperatura(20))   # Normal
print(clasificar_temperatura(24))   # Precaución
print(clasificar_temperatura(27))   # Alerta
print(clasificar_temperatura(None)) # Dato inválido

Normal
Precaución
Alerta
Dato inválido


## Parte 4 — Limpieza y procesamiento de registros

Procese la lista `registros_crudos` y cree una nueva lista llamada `registros_limpios`.

Cada registro limpio debe contener:

- `id`
- `voltaje`
- `corriente`
- `temperatura`
- `potencia`
- `estado_temperatura`

Use las funciones creadas.

In [6]:
# TODO: procese los registros

# Se usa astype(str) para simular datos "crudos" en texto (como llegarían de un
# sensor real), y así convertir_a_float() cumple su función de validación.
registros_crudos = df.astype(str).to_dict("records")

registros_limpios = []

for r in registros_crudos:
    id_registro = int(float(r["time"]))
    temperatura = convertir_a_float(r["temperature_C"])
    vibracion = convertir_a_float(r["vibration_g"])
    corriente = convertir_a_float(r["current_A"])

    estado_temperatura = clasificar_temperatura(temperatura)
    estado_vibracion = clasificar_vibracion(vibracion)

    registro_limpio = {
        "id": id_registro,
        "temperatura": temperatura,
        "vibracion": vibracion,
        "corriente": corriente,
        "estado_temperatura": estado_temperatura,
        "estado_vibracion": estado_vibracion,
    }

    registros_limpios.append(registro_limpio)

print(len(registros_limpios))
registros_limpios[:3]

1000


[{'id': 0,
  'temperatura': 20.397371322408983,
  'vibracion': 0.7099033154879003,
  'corriente': 1.9324821725025616,
  'estado_temperatura': 'Normal',
  'estado_vibracion': 'Precaución'},
 {'id': 1,
  'temperatura': 19.98938189252972,
  'vibracion': 0.6386950524369153,
  'corriente': 1.9955462811794744,
  'estado_temperatura': 'Normal',
  'estado_vibracion': 'Normal'},
 {'id': 2,
  'temperatura': 20.718097501413723,
  'vibracion': 0.5089445554880261,
  'corriente': 1.9407431963770216,
  'estado_temperatura': 'Normal',
  'estado_vibracion': 'Normal'}]

## Parte 5 — Comprensión de listas

Use comprensión de listas para obtener:

1. Lista de potencias válidas.
2. Lista de temperaturas válidas.
3. Lista de motores en alerta.

In [7]:
# TODO: use comprensión de listas

# 1. Lista de vibraciones válidas (descarta los None)
vibraciones_validas = [r["vibracion"] for r in registros_limpios if r["vibracion"] is not None]

# 2. Lista de temperaturas válidas (descarta los None)
temperaturas_validas = [r["temperatura"] for r in registros_limpios if r["temperatura"] is not None]

# 3. Lista de registros en alerta (ids donde temperatura o vibración están en "Alerta")
registros_alerta = [
    r["id"] for r in registros_limpios
    if r["estado_temperatura"] == "Alerta" or r["estado_vibracion"] == "Alerta"
]

print(len(vibraciones_validas))
print(len(temperaturas_validas))
print(len(registros_alerta))
print(registros_alerta)

1000
1000
12
[71, 73, 82, 374, 378, 387, 393, 397, 615, 707, 709, 957]


## Parte 6 — Comprensión de diccionarios

Cree un diccionario llamado `estado_por_motor` donde:

- La clave sea el `id` del motor.
- El valor sea el `estado_temperatura`.

In [8]:
# TODO: cree el diccionario estado_por_motor

# Diccionario donde la clave es el id del registro (el tiempo de la lectura)
# y el valor es su estado_temperatura, para poder consultar rápido
# el estado de cualquier registro por su id.
estado_por_registro = {r["id"]: r["estado_temperatura"] for r in registros_limpios}

print(len(estado_por_registro))
print(estado_por_registro[71])   # Alerta
print(estado_por_registro[0])    # Normal

1000
Alerta
Normal


## Parte 7 — Resumen automático

Cree una función llamada `generar_resumen(registros)` que retorne un diccionario con:

- `total_registros`
- `registros_validos_potencia`
- `potencia_promedio`
- `temperatura_promedio`
- `cantidad_alertas`
- `cantidad_precauciones`
- `cantidad_normales`

In [9]:
# TODO: cree la función generar_resumen

def generar_resumen(registros):
    """
    Calcula métricas generales sobre una lista de registros procesados:
    total de registros, cuántos tienen vibración válida, el promedio de
    vibración y de temperatura (usando solo valores no nulos), y la
    cantidad de registros en cada estado de temperatura.
    """
    total_registros = len(registros)

    vibraciones_validas = [r["vibracion"] for r in registros if r["vibracion"] is not None]
    temperaturas_validas = [r["temperatura"] for r in registros if r["temperatura"] is not None]

    registros_validos_vibracion = len(vibraciones_validas)

    vibracion_promedio = sum(vibraciones_validas) / len(vibraciones_validas) if vibraciones_validas else None
    temperatura_promedio = sum(temperaturas_validas) / len(temperaturas_validas) if temperaturas_validas else None

    cantidad_alertas = sum(1 for r in registros if r["estado_temperatura"] == "Alerta")
    cantidad_precauciones = sum(1 for r in registros if r["estado_temperatura"] == "Precaución")
    cantidad_normales = sum(1 for r in registros if r["estado_temperatura"] == "Normal")

    return {
        "total_registros": total_registros,
        "registros_validos_vibracion": registros_validos_vibracion,
        "vibracion_promedio": vibracion_promedio,
        "temperatura_promedio": temperatura_promedio,
        "cantidad_alertas": cantidad_alertas,
        "cantidad_precauciones": cantidad_precauciones,
        "cantidad_normales": cantidad_normales,
    }

resumen = generar_resumen(registros_limpios)
resumen

{'total_registros': 1000,
 'registros_validos_vibracion': 1000,
 'vibracion_promedio': 0.5106254355873734,
 'temperatura_promedio': 20.161157833395325,
 'cantidad_alertas': 10,
 'cantidad_precauciones': 270,
 'cantidad_normales': 720}

## Parte 8 — Visualización como tabla

Convierta `registros_limpios` en un `DataFrame` de Pandas.

In [10]:
import pandas as pd

# TODO: cree un DataFrame llamado df

# Convertimos registros_limpios (lista de diccionarios) en un DataFrame:
# cada diccionario se vuelve una fila y cada clave, una columna.
df_resultados = pd.DataFrame(registros_limpios)
df_resultados.head()

,id,temperatura,vibracion,corriente,estado_temperatura,estado_vibracion
0,0,20.397371,0.709903,1.932482,Normal,Precaución
1,1,19.989382,0.638695,1.995546,Normal,Normal
2,2,20.718098,0.508945,1.940743,Normal,Normal
3,3,21.518244,0.402959,1.999154,Normal,Normal
4,4,20.212251,0.604733,1.850520,Normal,Normal


## Parte 9 — Buenas prácticas

Revise su notebook y verifique:

- Los nombres de variables son claros.
- Las funciones tienen una tarea específica.
- El código no está repetido innecesariamente.
- Las celdas están ejecutadas en orden.
- Hay explicaciones en texto.

## Parte 10 — Repositorio inicial en GitHub

Cree un repositorio en GitHub con el nombre:

**analisis-datos-python-apellido-nombre**

Estructura mínima esperada:

```text
analisis-datos-python-apellido-nombre/
│
├── README.md
├── requirements.txt
├── notebooks/
│   ├── semana_02_introduccion.ipynb
│   ├── semana_03_fundamentos_python.ipynb
│   └── semana_04_funciones_modularizacion.ipynb
│
├── data/
│   ├── raw/
│   └── processed/
│
├── src/
│   └── funciones.py
│
└── reports/
    └── figuras/
```

Enlace al repositorio:

Pegue aquí el enlace de su repositorio. https://github.com/KIZZYE/analisis-datos-python-hidalgo-kizzye

**Enlace al repositorio:**  
Escriba aquí el enlace.

## Parte 11 — Reflexión final

Responda:

1. ¿Qué ventaja tuvo usar funciones en el procesamiento de los registros?
2. ¿Por qué es importante manejar errores antes de analizar datos?
3. ¿Qué función del taller considera más útil?
4. ¿Qué elemento del repositorio ayuda más a la reproducibilidad?

**Respuesta:**  
1. Usar funciones permitió aplicar la misma lógica de conversión y clasificación a los 1000 registros del dataset sin repetir código; bastó llamarlas dentro del for y si había que ajustar un umbral (como pasó con clasificar_temperatura), solo se cambiaba en un solo lugar

2. Manejar errores antes de analizar es clave porque aunque este dataset venía limpio en la práctica los sensores pueden fallar y entregar texto vacío, None o valores fuera de rango; sin convertir_a_float capturando esos casos, el análisis se detendría con una excepción en vez de seguir procesando el resto de los registros

3. La más útil fue clasificar_temperatura (y su equivalente clasificar_vibracion), porque los umbrales que definimos ahí terminaron replicando exactamente los 12 registros que el dataset real marca como "alerta" mostró que una función bien calibrada puede detectar fallas reales del sistema

4. El README.md sigue siendo el que más ayuda a la reproducibilidad, porque documenta qué contiene el repositorio y qué herramientas se usaron, para que cualquier persona pueda entender y correr el análisis sin adivinar la estructura